# CAI 5335 Final Project Colab file
Siddharth Sivaram, Chandan Avireni
Dr. Anshuman Chhabra


##You can run all experiments here in this colab file. Make sure you have access to huggingface models relevant for this project. The models can be found below:

https://huggingface.co/mistralai/Mistral-7B-v0.1

https://huggingface.co/Qwen/Qwen2.5-14B/blob/main/README.md

https://huggingface.co/google/gemma-7b

https://huggingface.co/meta-llama/Llama-2-7b

https://huggingface.co/meta-llama/Llama-2-13b

https://huggingface.co/meta-llama/Meta-Llama-3-8B


The github files and repo for the code we use is found here (This project is open-source under the MIT License):
https://github.com/JieSLi/LLM-passphrase/tree/master


Link to the original paper to run experiments and replicate: https://aclanthology.org/2025.findings-naacl.290.pdf


Simply follow the instructions for each cell and run. It is beneficial to have the A100 GPU wiwth high RAM or h100 for computational speed and bandwith.





### Clone the github repo and relevant files.

We will use this files to manage prompts, run models and make overwite changes to. The files will be saved under file LLM-passphrase in the colab disk

In [2]:
!git clone https://github.com/JieSLi/LLM-passphrase.git
%cd LLM-passphrase
!pip install -q transformers accelerate bitsandbytes sentencepiece huggingface_hub

Cloning into 'LLM-passphrase'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 11 (delta 1), reused 11 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), 9.29 KiB | 9.29 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/LLM-passphrase
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.5 MB/s eta 0:00:00


### Set the HF_HOME Environment Variable

This is where the model will be downloaaded and stored

In [3]:
import os

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache"

### Log into Huggingface to grant access.

You need to generate a token with appropriate public permissions

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Tweaks for our projects

Adding Qwen model and changing it to load in 4bit for better computational efficieny in colab



#### Overwite existing files with our additions to load models easier and add qwen

In [15]:
%%writefile model_manager.py
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os
import torch

MODEL_PATHS = {
    "Llama-2-7b": "meta-llama/Llama-2-7b-chat-hf",
    "Llama-2-13b": "meta-llama/Llama-2-13b-chat-hf",
    "Llama-2-70b": "meta-llama/Llama-2-70b-chat-hf",
    "Llama-3-8B": "meta-llama/Meta-Llama-3-8B-Instruct",
    "Llama-3-70B": "meta-llama/Meta-Llama-3-70B-Instruct",
    "gemma-7b": "google/gemma-7b-it",
    "Mistral-7B": "mistralai/Mistral-7B-Instruct-v0.2",
    "Mixtral-8x7B": "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen2.5-14B": "Qwen/Qwen2.5-14B-Instruct",
}

def load_model_and_tokenizer(args_model, load_in_4bit=True):
    if args_model not in MODEL_PATHS:
        raise ValueError(f"Unknown model: {args_model}")

    model_path = MODEL_PATHS[args_model]
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = None
    if load_in_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        dtype=torch.float16,
        quantization_config=quantization_config,
    ).eval()

    model_name = os.path.basename(model_path.rstrip("/"))
    return model_name, model, tokenizer

    ##Commented for easy check to see if code actually changed

Overwriting model_manager.py


In [16]:
%%writefile args_parser.py
import argparse


def parse_arguments():

    parser = argparse.ArgumentParser(
        description="Generate text from selected language model."
    )
    parser.add_argument(
        "--in_prompt",
        default=6,
        help="enter either 1,2,3,... as number of examples in conv or enter input file as string",
    )
    parser.add_argument(
        "--out_subdir",
        type=str,
        default="test",
        help="out_subdir name (not whole path)",
    )
    parser.add_argument(
        "--bsz",
        type=int,
        default=16,
        help="batch size. On 2 A5000s, 16 works for all.",
    )
    parser.add_argument("--temp_list", type=float, nargs="+", help="list of temps")
    parser.add_argument("--top_p_list", type=float, nargs="+", help="list of top_ps")
    parser.add_argument(
        "--bot_q",
        type=float,
        default=0.0,
        help="this is top-p truncation value",
    )
    parser.add_argument("--N", type=int, default=1, help="number of iterations")
    parser.add_argument("--sd", type=int, default=100, help="random seed")
    parser.add_argument("--red_list", action="store_true", help="red_list")
    parser.add_argument("--end_token_ids", action="store_true", help="end_token_ids")
    parser.add_argument(
        "--model",
        type=str,
        choices=[
            "Llama-2-7b",
            "Llama-2-13b",
            "Llama-2-70b",
            "Llama-3-8B",
            "Llama-3-70B",
            "gemma-7b",
            "Mistral-7B",
            "Mixtral-8x7B",
            "Qwen2.5-14B",
        ],
        default="Llama-2-7b",
        help="Choose a language model among Llama-2-7b, Llama-2-13b, Llama-2-70b, Llama-3-8B, Llama-3-70B, Mistral-7B, Mixtral-8x7B, gemma-7b. Default is 'Llama-2-7b' when no arg is provided.",
    )
    return parser.parse_args()

    ##Commented for easy check to see if code actually changed


Overwriting args_parser.py


#### Inference file changed for safer handling and error mitigation

In [17]:
%%writefile main_sample.py
import os
import pickle

import torch
import torch.nn.functional as F

import pw_utils
from args_parser import parse_arguments
from model_manager import load_model_and_tokenizer
from prompt_manager import create_convs


def process_out(input_toks, tokenizer, generated_toks, out_ents):
    EOS = tokenizer.eos_token_id
    bsz, in_len = input_toks.shape

    out = generated_toks[:, in_len:]

    eos_n_1 = ((out == EOS).cumsum(1).cumsum(1) == 1).to(torch.uint8).argsort(1)[:, -1] + 1

    return [
        (
            row[:ind].tolist(),
            ents_row[:ind].tolist(),
            tokenizer.decode(row[:ind]),
        )
        for row, ents_row, ind in zip(out, out_ents, eos_n_1)
    ]


@torch.inference_mode()
def generate_toks(
    input_toks,
    model,
    tokenizer,
    red_list=None,
    end_token_ids=None,
    max_num_toks=50,
    temperature=1.0,
    bot_q=0.0,
    top_p=1.0,
    seed=None,
):
    EOS = tokenizer.eos_token_id

    torch.manual_seed(seed)
    bsz, prompt_len = input_toks.shape
    eos_flag = torch.zeros(bsz, 1, dtype=torch.bool, device=input_toks.device)
    out_ents = torch.empty(bsz, 0, device=input_toks.device)

    for inx in range(max_num_toks):
        logits = model(input_toks).logits[:, -1, :]

        if red_list:
            logits[:, red_list] = -float("inf")

        probs = F.softmax(logits / temperature, dim=-1)

        nan_check = torch.isnan(probs)
        if nan_check.any():
            rows_with_nan = nan_check.any(dim=1)
            rows_with_nan_indices = torch.nonzero(rows_with_nan).squeeze()

            if rows_with_nan_indices.dim() == 0:
                rows_with_nan_indices = rows_with_nan_indices.unsqueeze(0)

            for idx in rows_with_nan_indices:
                probs[idx] = 0
                probs[idx, EOS] = 1.0

        next_tok, next_tok_ent = pw_utils.pw_sample_top_p(
            probs, bot_q, top_p, EOS, end_token_ids
        )

        input_toks = torch.cat((input_toks, next_tok), dim=1)
        out_ents = torch.cat((out_ents, next_tok_ent), dim=1)

        if (next_tok == EOS).any():
            eos_flag[next_tok == EOS] = True

        if eos_flag.all():
            break

    return input_toks, out_ents


def main():
    args = parse_arguments()

    model_name, model, tokenizer = load_model_and_tokenizer(args.model)
    print("---- using model", model_name)

    red_list = None
    if args.red_list:
        with open("./red_lists/" + model_name + "_red_list.pkl", "rb") as f:
            red_list = pickle.load(f)

    end_token_ids = None
    if args.end_token_ids:
        with open("./red_lists/" + model_name + "_end_token_list.pkl", "rb") as f:
            end_token_ids = pickle.load(f)

    convs, quote_label = create_convs(str(args.in_prompt))

    sd = args.sd
    bsz = args.bsz

    if args.out_subdir is None:
        out_dir = f"./output_dir/{model_name}_{os.path.splitext(str(args.in_prompt))[0]}"
    else:
        out_dir = f"./output_dir/{args.out_subdir}"

    temp_list = [1.0, 1.2, 1.4]
    if args.temp_list:
        temp_list = args.temp_list

    top_p_list = [0.95, 0.99, 1.0]
    if args.top_p_list:
        top_p_list = args.top_p_list

    bot_q = args.bot_q
    t_p_grid = [(t, p) for t in temp_list for p in top_p_list]
    N = args.N

    for conv_idx, conv in enumerate(convs):
        if not str(args.in_prompt).isdigit():
            quote_label = conv[-1]["content"]

        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        for temperature, top_p in t_p_grid:
            for i in range(N):
                generated_toks, out_ents = generate_toks(
                    input_toks,
                    model,
                    tokenizer,
                    red_list,
                    end_token_ids,
                    temperature=temperature,
                    bot_q=bot_q,
                    top_p=top_p,
                    seed=sd + i,
                )

                out = process_out(input_toks, tokenizer, generated_toks, out_ents)

                pw_utils.logger(
                    tokenizer,
                    out_dir,
                    model_name,
                    in_str=quote_label,
                    toks_ents_str=out,
                    temperature=temperature,
                    top_p=top_p,
                    bot_q=bot_q,
                )

        pw_utils.log_args(
            out_dir,
            model_name,
            quote_label,
            args,
            temp_list,
            top_p_list,
        )


if __name__ == "__main__":
    main()

##Commented for easy check to see if code actually changed

Overwriting main_sample.py


####Some tests to run to check if set up worked

First is loading Qwen model to see if it downloaded and loaded weights

Second is running sample inference using mistral 7b model

These should run without error

In [18]:
import importlib
import model_manager
importlib.reload(model_manager)

from model_manager import load_model_and_tokenizer

In [19]:
model_name, model, tokenizer = load_model_and_tokenizer("Qwen2.5-14B")
print(model_name)

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Qwen2.5-14B-Instruct


####test: run inference on a model: mistral 7b

In [20]:
!python main_sample.py \
  --model "Mistral-7B" \
  --out_subdir "Mistral-7B_table3" \
  --bsz 1 \
  --N 5 \
  --in_prompt 6 \
  --temp_list 1.4 \
  --top_p_list 0.95 \
  --bot_q 0.0

config.json: 100% 596/596 [00:00<00:00, 2.95MB/s]
tokenizer_config.json: 2.10kB [00:00, 6.91MB/s]
tokenizer.json: 1.80MB [00:00, 134MB/s]
tokenizer.model: 100% 493k/493k [00:01<00:00, 483kB/s] 
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.86MB/s]
model.safetensors.index.json: 25.1kB [00:00, 70.5MB/s]
Fetching 3 files: 100% 3/3 [00:33<00:00, 11.29s/it]
Download complete: 100% 14.5G/14.5G [00:33<00:00, 427MB/s]
Loading weights: 100% 291/291 [00:04<00:00, 60.74it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 111/111 [00:00<00:00, 796kB/s]
---- using model Mistral-7B-Instruct-v0.2


In [21]:
!ls output_dir/Mistral-7B_table3

Mistral-7B-Instruct-v0.2_6_exs_2026-05-04_11-50-29.625688.json
Mistral-7B-Instruct-v0.2_6_exs_2026-05-04_11-50-30.746608.json
Mistral-7B-Instruct-v0.2_6_exs_2026-05-04_11-50-31.956694.json
Mistral-7B-Instruct-v0.2_6_exs_2026-05-04_11-50-32.887209.json
Mistral-7B-Instruct-v0.2_6_exs_2026-05-04_11-50-33.904542.json
Mistral-7B-Instruct-v0.2_6_exs__args_log_2026_05_04.txt


In [22]:
!head -c 1000 output_dir/Mistral-7B_table3/*.json

==> output_dir/Mistral-7B_table3/Mistral-7B-Instruct-v0.2_6_exs_2026-05-04_11-50-29.625688.json <==
{"num_entries": 1, "0": {"model": "Mistral-7B-Instruct-v0.2", "quote": "6_exs", "out_tokens": [10776, 286, 19808, 15853, 659, 5063, 1973, 5097, 2260, 2], "out_tokens_ent": [13.97, 0.69, 5.22, 13.46, 12.42, 10.71, 4.28, 13.55, 5.33, 1.36], "sent_ent": 81.0, "out_sent": "fooled fortune glance has torped progressively</s>", "num_words": 6, "temp": 1.4, "top_p": 0.95, "bot_q": 0.0, "id_tok_ent": [[10776, "\u2581fool", 13.97], [286, "ed", 0.69], [19808, "\u2581fortune", 5.22], [15853, "\u2581glance", 13.46], [659, "\u2581has", 12.42], [5063, "\u2581tor", 10.71], [1973, "ped", 4.28], [5097, "\u2581progress", 13.55], [2260, "ively", 5.33], [2, "</s>", 1.36]]}}
==> output_dir/Mistral-7B_table3/Mistral-7B-Instruct-v0.2_6_exs_2026-05-04_11-50-30.746608.json <==
{"num_entries": 1, "0": {"model": "Mistral-7B-Instruct-v0.2", "quote": "6_exs", "out_tokens": [15296, 9365, 22834, 3115, 11532, 1139, 2218

#### generate data table from the jsons and sample filtering criteria

In [23]:
import os, json, re
import pandas as pd

def clean_generation(text: str) -> str:
    text = text.replace("</s>", "").strip()
    text = re.sub(r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*", "", text, flags=re.IGNORECASE)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""
    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = text.replace("</s>", "").strip()
    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False
    leftover = re.sub(r"[A-Za-z\s'\-.,]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir"):
    if "table3" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        with open(os.path.join(root, file), "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)
            entropy = item.get("sent_ent")

            if entropy is None:
                continue

            n_words = count_words(text)
            model = item.get("model", os.path.basename(root).replace("_table3", ""))

            rows.append({
                "model": model,
                "text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table3 = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table3

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Mistral-7B-Instruct-v0.2,1.0,1.0,1.0,1.0


## Experiment 1: Entropy for models using baseline prompt

This experiment will test: Proportion of output that meet criteria of entropy ≥ 47.0, num_words ≤ 8 and consisting of all
English words, separately, and all criteria concurrently
using the base prompt, with temperature and top-p values manipulated. Total of 9 experiments run with top-p values of 0.95, 0.99 and 1.0 and temperature of 1.0, 1.2, 1.4

### Load necessary models and utils file

In [12]:
import os, gc, torch
import pw_utils

from model_manager import load_model_and_tokenizer
from prompt_manager import create_convs
from main_sample import generate_toks, process_out

### Define models to be used

In [ ]:
models = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]


### Experiment 1.1

Temperature of 1 and top-p of 0.95

In [29]:
temperature = 1.0
top_p = 0.95
bot_q = 0.0
in_prompt = "6"

In [30]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp2/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Generating 100 samples for Qwen2.5-14B...
Qwen2.5-14B: 10/100
Qwen2.5-14B: 20/100
Qwen2.5-14B: 30/100
Qwen2.5-14B: 40/100
Qwen2.5-14B: 50/100
Qwen2.5-14B: 60/100
Qwen2.5-14B: 70/100
Qwen2.5-14B: 80/100
Qwen2.5-14B: 90/100
Qwen2.5-14B: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Generating 100 samples for Mistral-7B...
Mistral-7B: 10/100
Mistral-7B: 20/100
Mistral-7B: 30/100
Mistral-7B: 40/100
Mistral-7B: 50/100
Mistral-7B: 60/100
Mistral-7B: 70/100
Mistral-7B: 80/100
Mistral-7B: 90/100
Mistral-7B: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf
Generating 100 samples for Llama-2-7b...
Llama-2-7b: 10/100
Llama-2-7b: 20/100
Llama-2-7b: 30/100
Llama-2-7b: 40/100
Llama-2-7b: 50/100
Llama-2-7b: 60/100
Llama-2-7b: 70/100
Llama-2-7b: 80/100
Llama-2-7b: 90/100
Llama-2-7b: 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct
Generating 100 samples for Llama-3-8B...
Llama-3-8B: 10/100
Llama-3-8B: 20/100
Llama-3-8B: 30/100
Llama-3-8B: 40/100
Llama-3-8B: 50/100
Llama-3-8B: 60/100
Llama-3-8B: 70/100
Llama-3-8B: 80/100
Llama-3-8B: 90/100
Llama-3-8B: 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf
Generating 100 samples for Llama-2-13b...
Llama-2-13b: 10/100
Llama-2-13b: 20/100
Llama-2-13b: 30/100
Llama-2-13b: 40/100
Llama-2-13b: 50/100
Llama-2-13b: 60/100
Llama-2-13b: 70/100
Llama-2-13b: 80/100
Llama-2-13b: 90/100
Llama-2-13b: 100/100
Finished Llama-2-13b


In [31]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

#filtering for clean output (removes prompt leakage, bad characters, end tokens etc)
def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)

    # Remove end tokens
    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()

    # Remove filler
    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove leakage
    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # only first line (removes Chinese + junk)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False

    # Allow normal English letters, spaces, apostrophes, hyphens, and punctuation.
    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp2"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.00,1.0,0.98,0.00
Mistral-7B,0.42,1.0,1.00,0.42
gemma-7b,0.26,0.9,0.93,0.22
Llama-2-7b,0.09,1.0,1.00,0.09
Llama-3-8B,0.12,1.0,1.00,0.12
Llama-2-13b,0.10,1.0,1.00,0.10


### Experiment 1.2

Temperature of 1 and top-p of 0.99

In [113]:
in_prompt = "6"
temperature = 1.0
top_p = 0.99
bot_q = 0.0

In [114]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp3/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Generating 100 samples for Qwen2.5-14B...
Qwen2.5-14B: 10/100
Qwen2.5-14B: 20/100
Qwen2.5-14B: 30/100
Qwen2.5-14B: 40/100
Qwen2.5-14B: 50/100
Qwen2.5-14B: 60/100
Qwen2.5-14B: 70/100
Qwen2.5-14B: 80/100
Qwen2.5-14B: 90/100
Qwen2.5-14B: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Generating 100 samples for Mistral-7B...
Mistral-7B: 10/100
Mistral-7B: 20/100
Mistral-7B: 30/100
Mistral-7B: 40/100
Mistral-7B: 50/100
Mistral-7B: 60/100
Mistral-7B: 70/100
Mistral-7B: 80/100
Mistral-7B: 90/100
Mistral-7B: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf
Generating 100 samples for Llama-2-7b...
Llama-2-7b: 10/100
Llama-2-7b: 20/100
Llama-2-7b: 30/100
Llama-2-7b: 40/100
Llama-2-7b: 50/100
Llama-2-7b: 60/100
Llama-2-7b: 70/100
Llama-2-7b: 80/100
Llama-2-7b: 90/100
Llama-2-7b: 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct
Generating 100 samples for Llama-3-8B...
Llama-3-8B: 10/100
Llama-3-8B: 20/100
Llama-3-8B: 30/100
Llama-3-8B: 40/100
Llama-3-8B: 50/100
Llama-3-8B: 60/100
Llama-3-8B: 70/100
Llama-3-8B: 80/100
Llama-3-8B: 90/100
Llama-3-8B: 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf
Generating 100 samples for Llama-2-13b...
Llama-2-13b: 10/100
Llama-2-13b: 20/100
Llama-2-13b: 30/100
Llama-2-13b: 40/100
Llama-2-13b: 50/100
Llama-2-13b: 60/100
Llama-2-13b: 70/100
Llama-2-13b: 80/100
Llama-2-13b: 90/100
Llama-2-13b: 100/100
Finished Llama-2-13b


In [115]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)


    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()


    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )


    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )


    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False


    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp3"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.00,1.0,0.98,0.00
Mistral-7B,0.58,1.0,1.00,0.58
gemma-7b,0.39,0.9,0.91,0.29
Llama-2-7b,0.12,1.0,1.00,0.12
Llama-3-8B,0.22,1.0,1.00,0.22
Llama-2-13b,0.21,1.0,1.00,0.21


### Experiment 1.3

Temperature of 1 and top-p of 1.0

In [110]:
in_prompt = "6"
temperature = 1.0
top_p = 1.0
bot_q = 0.0

In [111]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp4/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [42]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)


    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()


    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )


    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )


    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False


    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp4"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.00,1.00,0.98,0.00
Mistral-7B,0.64,1.00,1.00,0.64
gemma-7b,0.44,0.89,0.91,0.31
Llama-2-7b,0.18,1.00,1.00,0.18
Llama-3-8B,0.27,1.00,1.00,0.27
Llama-2-13b,0.26,1.00,0.99,0.25


### Experiment 1.4

Temperature of 1.2 and top-p of 0.95

In [27]:
in_prompt = "6"
temperature = 1.2
top_p = 0.95
bot_q = 0.0


In [28]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp5/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Generating 100 samples for Qwen2.5-14B...
Qwen2.5-14B: 10/100
Qwen2.5-14B: 20/100
Qwen2.5-14B: 30/100
Qwen2.5-14B: 40/100
Qwen2.5-14B: 50/100
Qwen2.5-14B: 60/100
Qwen2.5-14B: 70/100
Qwen2.5-14B: 80/100
Qwen2.5-14B: 90/100
Qwen2.5-14B: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Generating 100 samples for Mistral-7B...
Mistral-7B: 10/100
Mistral-7B: 20/100
Mistral-7B: 30/100
Mistral-7B: 40/100
Mistral-7B: 50/100
Mistral-7B: 60/100
Mistral-7B: 70/100
Mistral-7B: 80/100
Mistral-7B: 90/100
Mistral-7B: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Loaded: Llama-2-13b-chat-hf
Generating 100 samples for Llama-2-13b...
Llama-2-13b: 10/100
Llama-2-13b: 20/100
Llama-2-13b: 30/100
Llama-2-13b: 40/100
Llama-2-13b: 50/100
Llama-2-13b: 60/100
Llama-2-13b: 70/100
Llama-2-13b: 80/100
Llama-2-13b: 90/100
Llama-2-13b: 100/100
Finished Llama-2-13b


In [29]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)


    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()


    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )


    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )


    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False


    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp5"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.01,1.00,0.95,0.00
Mistral-7B,0.83,1.00,0.99,0.82
gemma-7b,0.63,0.94,0.88,0.49
Llama-2-7b,0.38,1.00,1.00,0.38
Llama-3-8B,0.48,0.99,0.98,0.46
Llama-2-13b,0.52,1.00,0.98,0.50


### Experiment 1.5
Temperature of 1.2 and top-p of 0.99

In [32]:
in_prompt = "6"
temperature = 1.2
top_p = 0.99
bot_q = 0.0


In [34]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp6/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Generating 100 samples for Qwen2.5-14B...
Qwen2.5-14B: 10/100
Qwen2.5-14B: 20/100
Qwen2.5-14B: 30/100
Qwen2.5-14B: 40/100
Qwen2.5-14B: 50/100
Qwen2.5-14B: 60/100
Qwen2.5-14B: 70/100
Qwen2.5-14B: 80/100
Qwen2.5-14B: 90/100
Qwen2.5-14B: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Generating 100 samples for Mistral-7B...
Mistral-7B: 10/100
Mistral-7B: 20/100
Mistral-7B: 30/100
Mistral-7B: 40/100
Mistral-7B: 50/100
Mistral-7B: 60/100
Mistral-7B: 70/100
Mistral-7B: 80/100
Mistral-7B: 90/100
Mistral-7B: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf
Generating 100 samples for Llama-2-7b...
Llama-2-7b: 10/100
Llama-2-7b: 20/100
Llama-2-7b: 30/100
Llama-2-7b: 40/100
Llama-2-7b: 50/100
Llama-2-7b: 60/100
Llama-2-7b: 70/100
Llama-2-7b: 80/100
Llama-2-7b: 90/100
Llama-2-7b: 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct
Generating 100 samples for Llama-3-8B...
Llama-3-8B: 10/100
Llama-3-8B: 20/100
Llama-3-8B: 30/100
Llama-3-8B: 40/100
Llama-3-8B: 50/100
Llama-3-8B: 60/100
Llama-3-8B: 70/100
Llama-3-8B: 80/100
Llama-3-8B: 90/100
Llama-3-8B: 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf
Generating 100 samples for Llama-2-13b...
Llama-2-13b: 10/100
Llama-2-13b: 20/100
Llama-2-13b: 30/100
Llama-2-13b: 40/100
Llama-2-13b: 50/100
Llama-2-13b: 60/100
Llama-2-13b: 70/100
Llama-2-13b: 80/100
Llama-2-13b: 90/100
Llama-2-13b: 100/100
Finished Llama-2-13b


In [35]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)


    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()


    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )


    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )


    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False


    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp6"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.02,1.00,0.94,0.00
Mistral-7B,0.86,0.98,0.97,0.82
gemma-7b,0.74,0.92,0.86,0.57
Llama-2-7b,0.49,1.00,0.99,0.48
Llama-3-8B,0.55,1.00,0.98,0.53
Llama-2-13b,0.66,1.00,0.96,0.62


### Experiment 1.6
Temperature of 1.2 and top-p of 1.0

In [37]:
in_prompt = "6"
temperature = 1.2
top_p = 1.0
bot_q = 0.0



In [38]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp7/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Generating 100 samples for Qwen2.5-14B...
Qwen2.5-14B: 10/100
Qwen2.5-14B: 20/100
Qwen2.5-14B: 30/100
Qwen2.5-14B: 40/100
Qwen2.5-14B: 50/100
Qwen2.5-14B: 60/100
Qwen2.5-14B: 70/100
Qwen2.5-14B: 80/100
Qwen2.5-14B: 90/100
Qwen2.5-14B: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Generating 100 samples for Mistral-7B...
Mistral-7B: 10/100
Mistral-7B: 20/100
Mistral-7B: 30/100
Mistral-7B: 40/100
Mistral-7B: 50/100
Mistral-7B: 60/100
Mistral-7B: 70/100
Mistral-7B: 80/100
Mistral-7B: 90/100
Mistral-7B: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf
Generating 100 samples for Llama-2-7b...
Llama-2-7b: 10/100
Llama-2-7b: 20/100
Llama-2-7b: 30/100
Llama-2-7b: 40/100
Llama-2-7b: 50/100
Llama-2-7b: 60/100
Llama-2-7b: 70/100
Llama-2-7b: 80/100
Llama-2-7b: 90/100
Llama-2-7b: 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct
Generating 100 samples for Llama-3-8B...
Llama-3-8B: 10/100
Llama-3-8B: 20/100
Llama-3-8B: 30/100
Llama-3-8B: 40/100
Llama-3-8B: 50/100
Llama-3-8B: 60/100
Llama-3-8B: 70/100
Llama-3-8B: 80/100
Llama-3-8B: 90/100
Llama-3-8B: 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf
Generating 100 samples for Llama-2-13b...
Llama-2-13b: 10/100
Llama-2-13b: 20/100
Llama-2-13b: 30/100
Llama-2-13b: 40/100
Llama-2-13b: 50/100
Llama-2-13b: 60/100
Llama-2-13b: 70/100
Llama-2-13b: 80/100
Llama-2-13b: 90/100
Llama-2-13b: 100/100
Finished Llama-2-13b


In [39]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)


    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()


    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )


    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )


    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False


    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp7"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.03,1.00,0.93,0.00
Mistral-7B,0.87,0.96,0.90,0.76
gemma-7b,0.78,0.91,0.85,0.59
Llama-2-7b,0.55,1.00,0.97,0.52
Llama-3-8B,0.60,1.00,0.95,0.55
Llama-2-13b,0.66,1.00,0.95,0.61


### Experiment 1.7

Temperature of 1.4 and top-p of 0.95

In [41]:
in_prompt = "6"
temperature = 1.4
top_p = 0.95
bot_q = 0.0


In [42]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp8/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Generating 100 samples for Qwen2.5-14B...
Qwen2.5-14B: 10/100
Qwen2.5-14B: 20/100
Qwen2.5-14B: 30/100
Qwen2.5-14B: 40/100
Qwen2.5-14B: 50/100
Qwen2.5-14B: 60/100
Qwen2.5-14B: 70/100
Qwen2.5-14B: 80/100
Qwen2.5-14B: 90/100
Qwen2.5-14B: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Generating 100 samples for Mistral-7B...
Mistral-7B: 10/100
Mistral-7B: 20/100
Mistral-7B: 30/100
Mistral-7B: 40/100
Mistral-7B: 50/100
Mistral-7B: 60/100
Mistral-7B: 70/100
Mistral-7B: 80/100
Mistral-7B: 90/100
Mistral-7B: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf
Generating 100 samples for Llama-2-7b...
Llama-2-7b: 10/100
Llama-2-7b: 20/100
Llama-2-7b: 30/100
Llama-2-7b: 40/100
Llama-2-7b: 50/100
Llama-2-7b: 60/100
Llama-2-7b: 70/100
Llama-2-7b: 80/100
Llama-2-7b: 90/100
Llama-2-7b: 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct
Generating 100 samples for Llama-3-8B...
Llama-3-8B: 10/100
Llama-3-8B: 20/100
Llama-3-8B: 30/100
Llama-3-8B: 40/100
Llama-3-8B: 50/100
Llama-3-8B: 60/100
Llama-3-8B: 70/100
Llama-3-8B: 80/100
Llama-3-8B: 90/100
Llama-3-8B: 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf
Generating 100 samples for Llama-2-13b...
Llama-2-13b: 10/100
Llama-2-13b: 20/100
Llama-2-13b: 30/100
Llama-2-13b: 40/100
Llama-2-13b: 50/100
Llama-2-13b: 60/100
Llama-2-13b: 70/100
Llama-2-13b: 80/100
Llama-2-13b: 90/100
Llama-2-13b: 100/100
Finished Llama-2-13b


In [43]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)


    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()


    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )


    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )


    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False


    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp8"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.14,0.97,0.79,0.00
Mistral-7B,0.99,0.91,0.95,0.88
gemma-7b,0.97,0.83,0.70,0.57
Llama-2-7b,0.79,1.00,0.95,0.75
Llama-3-8B,0.87,0.94,0.89,0.76
Llama-2-13b,0.83,1.00,0.96,0.79


### Experiment 1.8

Temperature of 1.4 and top-p of 0.99

In [45]:
in_prompt = "6"
temperature = 1.4
top_p = 0.99
bot_q = 0.0

In [48]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp9.1/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Generating 100 samples for Qwen2.5-14B...
Qwen2.5-14B: 10/100
Qwen2.5-14B: 20/100
Qwen2.5-14B: 30/100
Qwen2.5-14B: 40/100
Qwen2.5-14B: 50/100
Qwen2.5-14B: 60/100
Qwen2.5-14B: 70/100
Qwen2.5-14B: 80/100
Qwen2.5-14B: 90/100
Qwen2.5-14B: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Generating 100 samples for Mistral-7B...
Mistral-7B: 10/100
Mistral-7B: 20/100
Mistral-7B: 30/100
Mistral-7B: 40/100
Mistral-7B: 50/100
Mistral-7B: 60/100
Mistral-7B: 70/100
Mistral-7B: 80/100
Mistral-7B: 90/100
Mistral-7B: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf
Generating 100 samples for Llama-2-7b...
Llama-2-7b: 10/100
Llama-2-7b: 20/100
Llama-2-7b: 30/100
Llama-2-7b: 40/100
Llama-2-7b: 50/100
Llama-2-7b: 60/100
Llama-2-7b: 70/100
Llama-2-7b: 80/100
Llama-2-7b: 90/100
Llama-2-7b: 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct
Generating 100 samples for Llama-3-8B...
Llama-3-8B: 10/100
Llama-3-8B: 20/100
Llama-3-8B: 30/100
Llama-3-8B: 40/100
Llama-3-8B: 50/100
Llama-3-8B: 60/100
Llama-3-8B: 70/100
Llama-3-8B: 80/100
Llama-3-8B: 90/100
Llama-3-8B: 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf
Generating 100 samples for Llama-2-13b...
Llama-2-13b: 10/100
Llama-2-13b: 20/100
Llama-2-13b: 30/100
Llama-2-13b: 40/100
Llama-2-13b: 50/100
Llama-2-13b: 60/100
Llama-2-13b: 70/100
Llama-2-13b: 80/100
Llama-2-13b: 90/100
Llama-2-13b: 100/100
Finished Llama-2-13b


In [49]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)


    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()


    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )


    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )


    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False


    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp9.1"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.19,0.97,0.74,0.01
Mistral-7B,0.99,0.86,0.87,0.80
gemma-7b,0.98,0.80,0.66,0.54
Llama-2-7b,0.85,0.98,0.88,0.73
Llama-3-8B,0.94,0.96,0.87,0.81
Llama-2-13b,0.87,0.96,0.93,0.80


### Experiment 1.9
Temperature of 1.4 and top-p of 1.0

In [54]:
in_prompt = "6"
temperature = 1.4
top_p = 1.0
bot_q = 0.0


In [55]:
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)
    out_dir = f"./output_dir_exp10/{model_key}_table3_final"

    for conv in convs:
        input_toks = tokenizer.apply_chat_template(
            conv,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if isinstance(input_toks, dict):
            input_toks = input_toks["input_ids"]
        elif hasattr(input_toks, "input_ids"):
            input_toks = input_toks.input_ids

        input_toks = input_toks.to(model.device)
        input_toks = input_toks.repeat(bsz, 1)

        print(f"Generating {N} samples for {model_key}...")

        for i in range(N):
            generated_toks, out_ents = generate_toks(
                input_toks,
                model,
                tokenizer,
                red_list=None,
                end_token_ids=None,
                temperature=temperature,
                bot_q=bot_q,
                top_p=top_p,
                seed=sd + i,
            )

            out = process_out(input_toks, tokenizer, generated_toks, out_ents)

            pw_utils.logger(
                tokenizer,
                out_dir,
                model_name,
                in_str=quote_label,
                toks_ents_str=out,
                temperature=temperature,
                top_p=top_p,
                bot_q=bot_q,
            )

            if (i + 1) % 10 == 0:
                print(f"{model_key}: {i + 1}/{N}")

    print(f"Finished {model_key}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Generating 100 samples for Qwen2.5-14B...
Qwen2.5-14B: 10/100
Qwen2.5-14B: 20/100
Qwen2.5-14B: 30/100
Qwen2.5-14B: 40/100
Qwen2.5-14B: 50/100
Qwen2.5-14B: 60/100
Qwen2.5-14B: 70/100
Qwen2.5-14B: 80/100
Qwen2.5-14B: 90/100
Qwen2.5-14B: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Generating 100 samples for Mistral-7B...
Mistral-7B: 10/100
Mistral-7B: 20/100
Mistral-7B: 30/100
Mistral-7B: 40/100
Mistral-7B: 50/100
Mistral-7B: 60/100
Mistral-7B: 70/100
Mistral-7B: 80/100
Mistral-7B: 90/100
Mistral-7B: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf
Generating 100 samples for Llama-2-7b...
Llama-2-7b: 10/100
Llama-2-7b: 20/100
Llama-2-7b: 30/100
Llama-2-7b: 40/100
Llama-2-7b: 50/100
Llama-2-7b: 60/100
Llama-2-7b: 70/100
Llama-2-7b: 80/100
Llama-2-7b: 90/100
Llama-2-7b: 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct
Generating 100 samples for Llama-3-8B...
Llama-3-8B: 10/100
Llama-3-8B: 20/100
Llama-3-8B: 30/100
Llama-3-8B: 40/100
Llama-3-8B: 50/100
Llama-3-8B: 60/100
Llama-3-8B: 70/100
Llama-3-8B: 80/100
Llama-3-8B: 90/100
Llama-3-8B: 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf
Generating 100 samples for Llama-2-13b...
Llama-2-13b: 10/100
Llama-2-13b: 20/100
Llama-2-13b: 30/100
Llama-2-13b: 40/100
Llama-2-13b: 50/100
Llama-2-13b: 60/100
Llama-2-13b: 70/100
Llama-2-13b: 80/100
Llama-2-13b: 90/100
Llama-2-13b: 100/100
Finished Llama-2-13b


In [56]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = [
    "</s>",
    "<|im_end|>",
    "<|endoftext|>",
    "<|eot_id|>",
    "<eos>",
]

def clean_generation(text: str) -> str:
    if text is None:
        return ""

    text = str(text)


    for tok in [
        "</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"
    ]:
        text = text.replace(tok, "")

    text = text.strip()


    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )


    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.IGNORECASE
    )


    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return ""

    return lines[0].strip("\"'` ")

def count_words(text):
    return len(re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))

def is_english_like(text):
    text = clean_generation(text)

    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
    if len(words) == 0:
        return False


    leftover = re.sub(r"[A-Za-z\s'\-.,!?;:]", "", text)
    return leftover.strip() == ""

rows = []

for root, dirs, files in os.walk("output_dir_exp10"):
    if "table3_final" not in root.lower():
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        with open(path, "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_text = item.get("out_sent", "")
            text = clean_generation(raw_text)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            n_words = count_words(text)

            rows.append({
                "model": model,
                "raw_text": raw_text,
                "clean_text": text,
                "entropy": float(entropy),
                "num_words": n_words,
                "entropy >= 47.0": float(entropy) >= 47.0,
                "num_words <= 8": n_words <= 8,
                "Eng": is_english_like(text),
            })

df = pd.DataFrame(rows)

df["All criteria"] = (
    df["entropy >= 47.0"]
    & df["num_words <= 8"]
    & df["Eng"]
)

table = df.groupby("model")[[
    "entropy >= 47.0",
    "num_words <= 8",
    "Eng",
    "All criteria"
]].mean().round(2)

table = table.reindex([m for m in MODEL_ORDER if m in table.index])

table.to_csv("final_entropy_baseline_comparison.csv")
df.to_csv("final_entropy_baseline_all_outputs.csv", index=False)

table

,entropy >= 47.0,num_words <= 8,Eng,All criteria
model,,,,
Qwen2.5-14B,0.23,0.97,0.71,0.01
Mistral-7B,0.99,0.84,0.86,0.79
gemma-7b,0.98,0.81,0.67,0.54
Llama-2-7b,0.89,0.97,0.86,0.75
Llama-3-8B,0.94,0.96,0.83,0.77
Llama-2-13b,0.89,0.95,0.89,0.78


## Experiment 2. Topic steering

Topic steering will also be explored by introducing prompt templates centered around specific themes (cat, econ, and sports) and comparing these against neutral prompts to evaluate how content influences generated passphrases.

Run each cell to explore the effects of tailoring the prompt for a specific response keyword

#### Load necessary modules for safety

In [ ]:
import os, gc, torch
import pw_utils

from model_manager import load_model_and_tokenizer
from prompt_manager import create_convs
from main_sample import generate_toks, process_out

In [ ]:
#Fixed parameters
temperature = 1.0
top_p = 0.95
bot_q = 0.0

models = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]


### Experiment 2.1: Cat

In [60]:


# base + cats prompts
prompts = {
    "base": "6",
    "cats": "606",
}



bsz = 1
N = 100
sd = 100

# run model
for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    for prompt_name, in_prompt in prompts.items():
        print(f"\n--- {model_key} | {prompt_name} ---")

        convs, quote_label = create_convs(in_prompt)

        out_dir = f"./output_dircats/{model_key}_table5_{prompt_name}"

        for conv in convs:
            input_toks = tokenizer.apply_chat_template(
                conv,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )


            if isinstance(input_toks, dict):
                input_toks = input_toks["input_ids"]
            elif hasattr(input_toks, "input_ids"):
                input_toks = input_toks.input_ids

            input_toks = input_toks.to(model.device)
            input_toks = input_toks.repeat(bsz, 1)

            for i in range(N):
                generated_toks, out_ents = generate_toks(
                    input_toks,
                    model,
                    tokenizer,
                    red_list=None,
                    end_token_ids=None,
                    temperature=temperature,
                    bot_q=bot_q,
                    top_p=top_p,
                    seed=sd + i,
                )

                out = process_out(input_toks, tokenizer, generated_toks, out_ents)

                pw_utils.logger(
                    tokenizer,
                    out_dir,
                    model_name,
                    in_str=prompt_name,  # 👈 label stored here
                    toks_ents_str=out,
                    temperature=temperature,
                    top_p=top_p,
                    bot_q=bot_q,
                )

                if (i + 1) % 25 == 0:
                    print(f"{model_key} ({prompt_name}): {i + 1}/{N}")


    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    print(f"Finished {model_key}")


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct

--- Qwen2.5-14B | base ---
Qwen2.5-14B (base): 25/100
Qwen2.5-14B (base): 50/100
Qwen2.5-14B (base): 75/100
Qwen2.5-14B (base): 100/100

--- Qwen2.5-14B | cats ---
Qwen2.5-14B (cats): 25/100
Qwen2.5-14B (cats): 50/100
Qwen2.5-14B (cats): 75/100
Qwen2.5-14B (cats): 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2

--- Mistral-7B | base ---
Mistral-7B (base): 25/100
Mistral-7B (base): 50/100
Mistral-7B (base): 75/100
Mistral-7B (base): 100/100

--- Mistral-7B | cats ---
Mistral-7B (cats): 25/100
Mistral-7B (cats): 50/100
Mistral-7B (cats): 75/100
Mistral-7B (cats): 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf

--- Llama-2-7b | base ---
Llama-2-7b (base): 25/100
Llama-2-7b (base): 50/100
Llama-2-7b (base): 75/100
Llama-2-7b (base): 100/100

--- Llama-2-7b | cats ---
Llama-2-7b (cats): 25/100
Llama-2-7b (cats): 50/100
Llama-2-7b (cats): 75/100
Llama-2-7b (cats): 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct

--- Llama-3-8B | base ---
Llama-3-8B (base): 25/100
Llama-3-8B (base): 50/100
Llama-3-8B (base): 75/100
Llama-3-8B (base): 100/100

--- Llama-3-8B | cats ---
Llama-3-8B (cats): 25/100
Llama-3-8B (cats): 50/100
Llama-3-8B (cats): 75/100
Llama-3-8B (cats): 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf

--- Llama-2-13b | base ---
Llama-2-13b (base): 25/100
Llama-2-13b (base): 50/100
Llama-2-13b (base): 75/100
Llama-2-13b (base): 100/100

--- Llama-2-13b | cats ---
Llama-2-13b (cats): 25/100
Llama-2-13b (cats): 50/100
Llama-2-13b (cats): 75/100
Llama-2-13b (cats): 100/100
Finished Llama-2-13b


In [65]:
import os, json, re
import pandas as pd

OUTPUT_DIR = "output_dircats"

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = ["</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"]

def clean(text):
    if text is None:
        return ""
    text = str(text)
    for t in END_TOKENS:
        text = text.replace(t, "")
    text = re.sub(r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*", "", text, flags=re.I)
    text = re.sub(r"Give the summary of a story.*", "", text, flags=re.I)
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[0] if lines else ""

def has_cat(text):
    return bool(re.search(r"\b(cat|cats)\b", text.lower()))

def has_purr(text):
    return bool(re.search(r"\b(purr|purring|purrs|feline|kitten|kittens)\b", text.lower()))

rows = []

for root, _, files in os.walk(OUTPUT_DIR):
    folder = os.path.basename(root).lower()

    if folder.endswith("_table5_base"):
        prompt = "base"
    elif folder.endswith("_table5_cats"):
        prompt = "cats"
    else:
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        with open(os.path.join(root, file), "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            model = MODEL_NAME_MAP.get(item.get("model"), item.get("model"))
            text = clean(item.get("out_sent", ""))

            rows.append({
                "model": model,
                "prompt": prompt,
                "cat": has_cat(text),
                "purr": has_purr(text),
            })

df = pd.DataFrame(rows)

table5 = df.groupby(["model", "prompt"])[["cat", "purr"]].mean().unstack("prompt")

for col in [("cat", "base"), ("purr", "base"), ("cat", "cats"), ("purr", "cats")]:
    if col not in table5.columns:
        table5[col] = pd.NA

table5 = table5[[("cat", "base"), ("purr", "base"), ("cat", "cats"), ("purr", "cats")]]
table5.columns = ["base_cat", "base_purr", "cats_cat", "cats_purr"]

table5 = table5.reindex([m for m in MODEL_ORDER if m in table5.index])
table5 = table5.round(2)

table5.to_csv("table5_cat_prompt_results.csv")

table5

,base_cat,base_purr,cats_cat,cats_purr
model,,,,
Qwen2.5-14B,0.07,0.00,1.00,0.03
Mistral-7B,0.02,0.01,0.79,0.24
gemma-7b,0.05,0.01,0.74,0.17
Llama-2-7b,0.06,0.09,0.87,0.15
Llama-3-8B,0.01,0.00,0.80,0.16
Llama-2-13b,0.01,0.02,0.74,0.24


### Experiment 2.2: econ

In [67]:


prompts = {
    "base": "6",
    "econ": "806",
}


bsz = 1
N = 100
sd = 100

for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    for prompt_name, in_prompt in prompts.items():
        print(f"\n--- {model_key} | {prompt_name} ---")

        convs, quote_label = create_convs(in_prompt)
        out_dir = f"./output_direcon/{model_key}_{prompt_name}"

        for conv in convs:
            input_toks = tokenizer.apply_chat_template(
                conv,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )

            if isinstance(input_toks, dict):
                input_toks = input_toks["input_ids"]
            elif hasattr(input_toks, "input_ids"):
                input_toks = input_toks.input_ids

            input_toks = input_toks.to(model.device).repeat(bsz, 1)

            for i in range(N):
                generated_toks, out_ents = generate_toks(
                    input_toks,
                    model,
                    tokenizer,
                    red_list=None,
                    end_token_ids=None,
                    temperature=temperature,
                    bot_q=bot_q,
                    top_p=top_p,
                    seed=sd + i,
                )

                out = process_out(input_toks, tokenizer, generated_toks, out_ents)

                pw_utils.logger(
                    tokenizer,
                    out_dir,
                    model_name,
                    in_str=prompt_name,
                    toks_ents_str=out,
                    temperature=temperature,
                    top_p=top_p,
                    bot_q=bot_q,
                )

                if (i + 1) % 25 == 0:
                    print(f"{model_key} ({prompt_name}): {i + 1}/{N}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    print(f"Finished {model_key}")


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct

--- Qwen2.5-14B | base ---
Qwen2.5-14B (base): 25/100
Qwen2.5-14B (base): 50/100
Qwen2.5-14B (base): 75/100
Qwen2.5-14B (base): 100/100

--- Qwen2.5-14B | econ ---
Qwen2.5-14B (econ): 25/100
Qwen2.5-14B (econ): 50/100
Qwen2.5-14B (econ): 75/100
Qwen2.5-14B (econ): 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2

--- Mistral-7B | base ---
Mistral-7B (base): 25/100
Mistral-7B (base): 50/100
Mistral-7B (base): 75/100
Mistral-7B (base): 100/100

--- Mistral-7B | econ ---
Mistral-7B (econ): 25/100
Mistral-7B (econ): 50/100
Mistral-7B (econ): 75/100
Mistral-7B (econ): 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf

--- Llama-2-7b | base ---
Llama-2-7b (base): 25/100
Llama-2-7b (base): 50/100
Llama-2-7b (base): 75/100
Llama-2-7b (base): 100/100

--- Llama-2-7b | econ ---
Llama-2-7b (econ): 25/100
Llama-2-7b (econ): 50/100
Llama-2-7b (econ): 75/100
Llama-2-7b (econ): 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct

--- Llama-3-8B | base ---
Llama-3-8B (base): 25/100
Llama-3-8B (base): 50/100
Llama-3-8B (base): 75/100
Llama-3-8B (base): 100/100

--- Llama-3-8B | econ ---
Llama-3-8B (econ): 25/100
Llama-3-8B (econ): 50/100
Llama-3-8B (econ): 75/100
Llama-3-8B (econ): 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf

--- Llama-2-13b | base ---
Llama-2-13b (base): 25/100
Llama-2-13b (base): 50/100
Llama-2-13b (base): 75/100
Llama-2-13b (base): 100/100

--- Llama-2-13b | econ ---
Llama-2-13b (econ): 25/100
Llama-2-13b (econ): 50/100
Llama-2-13b (econ): 75/100
Llama-2-13b (econ): 100/100
Finished Llama-2-13b


In [68]:
import os, json, re
import pandas as pd

OUTPUT_DIR = "output_direcon"

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = ["</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"]

def clean(text):
    if text is None:
        return ""
    text = str(text)
    for t in END_TOKENS:
        text = text.replace(t, "")
    text = re.sub(r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*", "", text, flags=re.I)
    text = re.sub(r"Give the summary of a story.*", "", text, flags=re.I)
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[0] if lines else ""

def has_econ(text):
    return bool(re.search(
        r"\b(economy|economic|economics|inflation|deflation|stock|stocks|market|markets|trade|trading|finance|financial|income|prices|consumer|industry)\b",
        text.lower()
    ))

rows = []

for root, _, files in os.walk(OUTPUT_DIR):
    folder = os.path.basename(root)

    if folder.endswith("_base"):
        prompt = "base"
    elif folder.endswith("_econ"):
        prompt = "econ"
    else:
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        with open(os.path.join(root, file), "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            model = MODEL_NAME_MAP.get(item.get("model"), item.get("model"))
            text = clean(item.get("out_sent", ""))

            rows.append({
                "model": model,
                "prompt": prompt,
                "econ": has_econ(text),
            })

df = pd.DataFrame(rows)

table = df.groupby(["model", "prompt"])["econ"].mean().unstack("prompt")


for col in ["base", "econ"]:
    if col not in table.columns:
        table[col] = pd.NA

table = table[["base", "econ"]]
table.columns = ["base_econ", "econ_econ"]

table = table.reindex([m for m in MODEL_ORDER if m in table.index])
table = table.round(2)

table.to_csv("table_econ_prompt_results.csv")

table

,base_econ,econ_econ
model,,
Qwen2.5-14B,0.00,0.12
Mistral-7B,0.01,0.08
gemma-7b,0.00,0.09
Llama-2-7b,0.00,0.14
Llama-3-8B,0.00,0.14
Llama-2-13b,0.00,0.07


### Experiment 2.3: adding sports (not included in original github file)

#### Overwrrite prompt manager file with just a baseline and sports prompt selection

In [79]:
%%writefile prompt_manager.py

def create_convs(inp):
    inp = str(inp)

    #Baselinne
    if inp.isdigit() and int(inp) < 100:

        conv = [
            {"role": "user", "content": "Give the summary of a story in six or fewer words."},
            {"role": "assistant", "content": "Multiple erasers banish innovation sparks"},

            {"role": "user", "content": "Give the summary of a story in six or fewer words."},
            {"role": "assistant", "content": "Hassled celebrity resists spaceship visit"},

            {"role": "user", "content": "Give the summary of a story in six or fewer words."},
            {"role": "assistant", "content": "fabulous doctor maintained patient dreams"},

            {"role": "user", "content": "Give the summary of a story in six or fewer words."},
            {"role": "assistant", "content": "television shoot overflows fruit boat capacity"},

            {"role": "user", "content": "Give the summary of a story in six or fewer words."},
            {"role": "assistant", "content": "Pavlov patronizes papaya pyramid king"},

            {"role": "user", "content": "Give the summary of a story in six or fewer words."},
            {"role": "assistant", "content": "courteous yeti plays quiet prodigy"},

            {"role": "user", "content": "Give the summary of a story in six or fewer words."},
        ]

        conv = conv[: int(inp) * 2 + 1]
        convs = [conv]
        quote_label = f"{inp}_exs"

        return convs, quote_label

    # Adding sports
    elif inp.isdigit() and int(inp) < 1000:

        conv = [
            {"role": "user", "content": "Give the summary of a story in eight or fewer words."},
            {"role": "assistant", "content": "Athlete discovers magic ball before final match"},

            {"role": "user", "content": "Give the summary of a story in eight or fewer words."},
            {"role": "assistant", "content": "Baseball bat unlocks forgotten stadium mystery"},

            {"role": "user", "content": "Give the summary of a story in eight or fewer words."},
            {"role": "assistant", "content": "Tennis racket saves team during championship crisis"},

            {"role": "user", "content": "Give the summary of a story in eight or fewer words."},
            {"role": "assistant", "content": "Young coach inspires athletes after painful defeat"},

            {"role": "user", "content": "Give the summary of a story in eight or fewer words."},
            {"role": "assistant", "content": "Lost soccer ball reveals ancient village secret"},

            {"role": "user", "content": "Give the summary of a story in eight or fewer words."},
            {"role": "assistant", "content": "Runner outraces storm to save teammates"},

            {"role": "user", "content": "Give the summary of a story in eight or fewer words."},
        ]

        conv = conv[: (int(inp) - 900) * 2 + 1]
        convs = [conv]
        quote_label = f"sports_{inp}_exs"

        return convs, quote_label

    # Error
    else:
        raise ValueError(f"Unsupported prompt id: {inp}")

Overwriting prompt_manager.py


In [80]:
import importlib
import prompt_manager
importlib.reload(prompt_manager)

<module 'prompt_manager' from '/content/LLM-passphrase/prompt_manager.py'>

#### Load modules and parameters again just for saftey

In [ ]:
import os, gc, torch
import pw_utils

from model_manager import load_model_and_tokenizer
from prompt_manager import create_convs
from main_sample import generate_toks, process_out

temperature = 1.0
top_p = 0.95
bot_q = 0.0

models = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]


In [81]:


prompts = {
    "base": "6",
    "sports": "906",
}


bsz = 1
N = 100
sd = 100

for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    for prompt_name, in_prompt in prompts.items():
        print(f"\n--- {model_key} | {prompt_name} ---")

        convs, quote_label = create_convs(in_prompt)
        out_dir = f"./output_dirsports/{model_key}_{prompt_name}"

        for conv in convs:
            input_toks = tokenizer.apply_chat_template(
                conv,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )

            if isinstance(input_toks, dict):
                input_toks = input_toks["input_ids"]
            elif hasattr(input_toks, "input_ids"):
                input_toks = input_toks.input_ids

            input_toks = input_toks.to(model.device).repeat(bsz, 1)

            for i in range(N):
                generated_toks, out_ents = generate_toks(
                    input_toks,
                    model,
                    tokenizer,
                    red_list=None,
                    end_token_ids=None,
                    temperature=temperature,
                    bot_q=bot_q,
                    top_p=top_p,
                    seed=sd + i,
                )

                out = process_out(input_toks, tokenizer, generated_toks, out_ents)

                pw_utils.logger(
                    tokenizer,
                    out_dir,
                    model_name,
                    in_str=prompt_name,
                    toks_ents_str=out,
                    temperature=temperature,
                    top_p=top_p,
                    bot_q=bot_q,
                )

                if (i + 1) % 25 == 0:
                    print(f"{model_key} ({prompt_name}): {i + 1}/{N}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    print(f"Finished {model_key}")


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct

--- Qwen2.5-14B | base ---
Qwen2.5-14B (base): 25/100
Qwen2.5-14B (base): 50/100
Qwen2.5-14B (base): 75/100
Qwen2.5-14B (base): 100/100

--- Qwen2.5-14B | sports ---
Qwen2.5-14B (sports): 25/100
Qwen2.5-14B (sports): 50/100
Qwen2.5-14B (sports): 75/100
Qwen2.5-14B (sports): 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2

--- Mistral-7B | base ---
Mistral-7B (base): 25/100
Mistral-7B (base): 50/100
Mistral-7B (base): 75/100
Mistral-7B (base): 100/100

--- Mistral-7B | sports ---
Mistral-7B (sports): 25/100
Mistral-7B (sports): 50/100
Mistral-7B (sports): 75/100
Mistral-7B (sports): 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf

--- Llama-2-7b | base ---
Llama-2-7b (base): 25/100
Llama-2-7b (base): 50/100
Llama-2-7b (base): 75/100
Llama-2-7b (base): 100/100

--- Llama-2-7b | sports ---
Llama-2-7b (sports): 25/100
Llama-2-7b (sports): 50/100
Llama-2-7b (sports): 75/100
Llama-2-7b (sports): 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct

--- Llama-3-8B | base ---
Llama-3-8B (base): 25/100
Llama-3-8B (base): 50/100
Llama-3-8B (base): 75/100
Llama-3-8B (base): 100/100

--- Llama-3-8B | sports ---
Llama-3-8B (sports): 25/100
Llama-3-8B (sports): 50/100
Llama-3-8B (sports): 75/100
Llama-3-8B (sports): 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf

--- Llama-2-13b | base ---
Llama-2-13b (base): 25/100
Llama-2-13b (base): 50/100
Llama-2-13b (base): 75/100
Llama-2-13b (base): 100/100

--- Llama-2-13b | sports ---
Llama-2-13b (sports): 25/100
Llama-2-13b (sports): 50/100
Llama-2-13b (sports): 75/100
Llama-2-13b (sports): 100/100
Finished Llama-2-13b


In [83]:
import os, json, re
import pandas as pd

OUTPUT_DIR = "output_dirsports"

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

END_TOKENS = ["</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"]

def clean(text):
    if text is None:
        return ""
    text = str(text)
    for tok in END_TOKENS:
        text = text.replace(tok, "")
    text = re.sub(r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*", "", text, flags=re.I)
    text = re.sub(r"Give the summary of a story.*", "", text, flags=re.I)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

def has_sports(text):
    return bool(re.search(
        r"\b(ball|balls|athlete|athletes|bat|bats|racket|rackets|tennis|soccer|baseball|basketball|football|coach|coaches|team|teams|runner|runners|stadium|match|matches|game|games|championship|teammate|teammates|player|players|sport|sports)\b",
        text.lower()
    ))

rows = []

for root, _, files in os.walk(OUTPUT_DIR):
    folder = os.path.basename(root).lower()

    if folder.endswith("_base"):
        prompt = "base"
    elif folder.endswith("_sports"):
        prompt = "sports"
    else:
        continue

    for file in files:
        if not file.endswith(".json"):
            continue

        with open(os.path.join(root, file), "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            model = MODEL_NAME_MAP.get(item.get("model"), item.get("model"))
            text = clean(item.get("out_sent", ""))

            rows.append({
                "model": model,
                "prompt": prompt,
                "sports_hit": has_sports(text),
                "text": text,
            })

df_sports = pd.DataFrame(rows)

summary = (
    df_sports.groupby(["model", "prompt"])["sports_hit"]
    .agg(total="count", sports_hits="sum")
    .reset_index()
)

summary["proportion"] = summary["sports_hits"] / summary["total"]

table_sports = summary.pivot(index="model", columns="prompt", values="proportion")

for col in ["base", "sports"]:
    if col not in table_sports.columns:
        table_sports[col] = pd.NA

table_sports = table_sports[["base", "sports"]]
table_sports.columns = ["base_sports", "sports_sports"]

table_sports = table_sports.reindex([m for m in MODEL_ORDER if m in table_sports.index])
table_sports = table_sports.round(2)

table_sports.to_csv("table_sports_prompt_results.csv")
df_sports.to_csv("table_sports_all_outputs.csv", index=False)

table_sports

,base_sports,sports_sports
model,,
Qwen2.5-14B,0.00,0.54
Mistral-7B,0.00,0.14
gemma-7b,0.01,0.11
Llama-2-7b,0.00,0.04
Llama-3-8B,0.00,0.21
Llama-2-13b,0.00,0.49


## Extension:: LLM as judge

Testing whether LLM's can serve as a judge and identify how difficiult or easy the previously generated passwords are.

Using mistral, llama 3-8b and gemma 7b as judges

### Extension 1. Topic steering vs baseline

In [103]:
# Download modules again for safety

import os, json, re, gc
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Access stored jsons and data from the topic steering cells
OUTPUT_DIRS = [
    "output_direcon",
    "output_dirsports",
    "output_dircats",
]

BASE_SAMPLE_SIZE = 100
TOPIC_SAMPLE_SIZE = 100

#Judges
JUDGE_MODELS = {
    "judge_mistral": "mistralai/Mistral-7B-Instruct-v0.2",
    "judge_llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
    "judge_gemma": "google/gemma-7b-it",
}

# Filtering code like previously used
END_TOKENS = ["</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"]

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

def clean(text):
    text = "" if text is None else str(text)

    for tok in END_TOKENS:
        text = text.replace(tok, "")

    text = re.sub(
        r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*",
        "",
        text,
        flags=re.I,
    )

    text = re.sub(
        r"Give the summary of a story.*",
        "",
        text,
        flags=re.I,
    )

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

# Load all data frrom directories
rows = []

for output_dir in OUTPUT_DIRS:
    if not os.path.exists(output_dir):
        print(f"Skipping missing folder: {output_dir}")
        continue

    for root, _, files in os.walk(output_dir):
        folder = os.path.basename(root).lower()

        if folder.endswith("_base"):
            prompt_group = "base"
            topic = "base"
        elif folder.endswith("_econ"):
            prompt_group = "topic_steered"
            topic = "econ"
        elif folder.endswith("_sports"):
            prompt_group = "topic_steered"
            topic = "sports"
        elif folder.endswith("_cats"):
            prompt_group = "topic_steered"
            topic = "cats"
        else:
            continue

        for file in files:
            if not file.endswith(".json"):
                continue

            path = os.path.join(root, file)

            with open(path, "r") as f:
                data = json.load(f)

            for k, item in data.items():
                if k == "num_entries":
                    continue

                raw_model = item.get("model")
                model = MODEL_NAME_MAP.get(raw_model, raw_model)

                rows.append({
                    "source_dir": output_dir,
                    "folder": os.path.basename(root),
                    "model": model,
                    "prompt_group": prompt_group,
                    "topic": topic,
                    "text": clean(item.get("out_sent", "")),
                    "entropy": item.get("sent_ent"),
                })

df_all = pd.DataFrame(rows)

if df_all.empty:
    raise ValueError("No outputs found. Check OUTPUT_DIRS and folder names.")

print(df_all["prompt_group"].value_counts())
print(df_all["topic"].value_counts())

# Sample 100 baseline and 100 topic ouyputs
df_base = df_all[df_all["prompt_group"] == "base"].sample(
    min(BASE_SAMPLE_SIZE, len(df_all[df_all["prompt_group"] == "base"])),
    random_state=42,
)

df_topic = df_all[df_all["prompt_group"] == "topic_steered"].sample(
    min(TOPIC_SAMPLE_SIZE, len(df_all[df_all["prompt_group"] == "topic_steered"])),
    random_state=42,
)

df = pd.concat([df_base, df_topic], ignore_index=True)

print("\nSample used:")
print(df["prompt_group"].value_counts())
print(df["topic"].value_counts())

# Prompt for the judge
JUDGE_PROMPT = """<s>[INST]
You are a hacker trying to guess the passphrase to log in.
Rate how difficult this passphrase is to guess.

Return ONLY:
Score: X

Where X is 1 to 5.

Passphrase: {text}
[/INST]"""

def score_with_judge(text, judge_model, judge_tokenizer):
    prompt = JUDGE_PROMPT.format(text=text)

    inputs = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)

    outputs = judge_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=judge_tokenizer.eos_token_id,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = judge_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    match = re.search(r"Score\s*:\s*([1-5])", decoded, flags=re.I)
    if not match:
        match = re.search(r"\b([1-5])\b", decoded)

    return int(match.group(1)) if match else None

# Run the judges
for judge_col, judge_model_name in JUDGE_MODELS.items():
    print(f"\n===== Loading {judge_col}: {judge_model_name} =====")

    judge_tokenizer = AutoTokenizer.from_pretrained(judge_model_name)

    if judge_tokenizer.pad_token is None:
        judge_tokenizer.pad_token = judge_tokenizer.eos_token

    judge_model = AutoModelForCausalLM.from_pretrained(
        judge_model_name,
        device_map="auto",
        torch_dtype=torch.float16,
    ).eval()

    scores = []

    for i, row in df.iterrows():
        text = row["text"]

        if not isinstance(text, str) or len(text.strip()) == 0:
            scores.append(None)
            continue

        score = score_with_judge(text, judge_model, judge_tokenizer)
        scores.append(score)

        if (i + 1) % 20 == 0:
            print(f"{judge_col}: {i + 1}/{len(df)}")

    df[judge_col] = scores

    del judge_model
    del judge_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

# Create table
judge_cols = list(JUDGE_MODELS.keys())

df["judge_avg"] = df[judge_cols].mean(axis=1).round(2)

final_table = (
    df.groupby("prompt_group")[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

final_table = final_table.reindex(["base", "topic_steered"])

# topic-specific breakdown
topic_table = (
    df.groupby("topic")[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

# model + prompt-group breakdown
model_table = (
    df.groupby(["model", "prompt_group"])[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

# Save tables and data
df.to_csv("topic_steering_judge_sample_200.csv", index=False)
final_table.to_csv("topic_steering_base_vs_topic_summary.csv")
topic_table.to_csv("topic_steering_by_topic_summary.csv")
model_table.to_csv("topic_steering_by_model_summary.csv")

print("\nBase vs topic-steered:")
display(final_table)

print("\nBy topic:")
display(topic_table)

print("\nBy model and prompt group:")
display(model_table)

prompt_group
topic_steered    1800
base             1800
Name: count, dtype: int64
topic
base      1800
econ       600
sports     600
cats       600
Name: count, dtype: int64

Sample used:
prompt_group
base             100
topic_steered    100
Name: count, dtype: int64
topic
base      100
cats       39
econ       39
sports     22
Name: count, dtype: int64

===== Loading judge_mistral: mistralai/Mistral-7B-Instruct-v0.2 =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

judge_mistral: 20/200
judge_mistral: 40/200
judge_mistral: 60/200
judge_mistral: 80/200
judge_mistral: 100/200
judge_mistral: 120/200
judge_mistral: 140/200
judge_mistral: 160/200
judge_mistral: 180/200
judge_mistral: 200/200

===== Loading judge_llama3: meta-llama/Meta-Llama-3-8B-Instruct =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

,judge_mistral,judge_llama3,judge_gemma,judge_avg
prompt_group,,,,
base,2.80,3.02,3.18,2.99
topic_steered,2.68,3.24,3.24,3.02



By topic:


,judge_mistral,judge_llama3,judge_gemma,judge_avg
topic,,,,
base,2.80,3.02,3.18,2.99
cats,2.51,3.10,3.15,2.88
econ,2.85,3.39,3.38,3.20
sports,2.68,3.12,3.14,2.95



By model and prompt group:


judge_mistral  judge_llama3  judge_gemma  judge_avg
model       prompt_group                                                      
Llama-2-13b base                    3.05          3.10         3.14       3.10
            topic_steered           2.60          3.27         3.13       2.97
Llama-2-7b  base                    2.64          3.00         3.09       2.89
            topic_steered           2.55          3.08         3.00       2.83
Llama-3-8B  base                    2.75          3.00         3.31       3.00
            topic_steered           2.64          3.21         3.44       3.09
Mistral-7B  base                    3.19          3.00         3.12       3.12
            topic_steered           2.93          3.33         3.36       3.17
Qwen2.5-14B base                    2.25          3.00         3.10       2.74
            topic_steered           2.64          3.12         3.18       2.94
gemma-7b    base                    2.94          3.00         3.31       3.09
            topic_steered           2.80          3.45         3.27       3.13

### Extension 2: test response over differing p top-p value

Using ou experiment 1.1, 1.2, and 1.3 jsons to test differing p value

In [119]:
#load modules again if running separately
import os, json, re, gc
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

#parametes and directories to pull
OUTPUT_DIRS = [
    "/content/content/LLM-passphrase/output_dir_exp2",
    "output_dir_exp3",
    "/content/content/LLM-passphrase/output_dir_exp4",
]

SAMPLE_SIZE = 200

JUDGE_MODELS = {
    "judge_mistral": "mistralai/Mistral-7B-Instruct-v0.2",
    "judge_llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
    "judge_gemma": "google/gemma-7b-it",
}

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

#filtering criteria
END_TOKENS = ["</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"]


def clean(text):
    text = "" if text is None else str(text)

    for tok in END_TOKENS:
        text = text.replace(tok, "")

    text = re.sub(r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*", "", text, flags=re.I)
    text = re.sub(r"Give the summary of a story.*", "", text, flags=re.I)

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

# Load data
rows = []

for output_dir in OUTPUT_DIRS:
    if not os.path.exists(output_dir):
        print(f"Skipping missing folder: {output_dir}")
        continue

    for root, _, files in os.walk(output_dir):
        for file in files:
            if not file.endswith(".json"):
                continue

            path = os.path.join(root, file)

            with open(path, "r") as f:
                data = json.load(f)

            for k, item in data.items():
                if k == "num_entries":
                    continue

                raw_model = item.get("model")
                model = MODEL_NAME_MAP.get(raw_model, raw_model)

                rows.append({
                    "source_dir": output_dir,
                    "folder": os.path.basename(root),
                    "model": model,
                    "text": clean(item.get("out_sent", "")),
                    "entropy": item.get("sent_ent"),
                    "temp": item.get("temp"),
                    "top_p": item.get("top_p"),
                    "bot_q": item.get("bot_q"),
                })

df_all = pd.DataFrame(rows)

if df_all.empty:
    raise ValueError("No JSON outputs found. Check OUTPUT_DIRS.")

print("Loaded rows by directory:")
print(df_all["source_dir"].value_counts())

print("\nDetected top_p values:")
print(df_all.groupby("source_dir")["top_p"].unique())

# Sample 50 dps per p value

samples = []

for exp_dir in OUTPUT_DIRS:
    df_subset = df_all[df_all["source_dir"] == exp_dir]

    if df_subset.empty:
        print(f"Skipping empty: {exp_dir}")
        continue

    sample_n = min(50, len(df_subset))

    sampled = df_subset.sample(sample_n, random_state=42)
    samples.append(sampled)

    print(f"{exp_dir}: sampled {sample_n}")

df = pd.concat(samples, ignore_index=True)

print("\nFinal sample distribution:")
print(df["source_dir"].value_counts())
print("\nTop-p per directory:")
print(df.groupby("source_dir")["top_p"].unique())

# Same prompt forr judge
JUDGE_PROMPT = """<s>[INST]
You are a hacker trying to guess the passphrase to log in.
Rate how difficult this passphrase is to guess.

Return ONLY:
Score: X

Where X is 1 to 5.

Passphrase: {text}
[/INST]"""

def score_with_judge(text, judge_model, judge_tokenizer):
    prompt = JUDGE_PROMPT.format(text=text)

    inputs = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)

    outputs = judge_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=judge_tokenizer.eos_token_id,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = judge_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    match = re.search(r"Score\s*:\s*([1-5])", decoded, flags=re.I)
    if not match:
        match = re.search(r"\b([1-5])\b", decoded)

    return int(match.group(1)) if match else None

# Run the judges
for judge_col, judge_model_name in JUDGE_MODELS.items():
    print(f"\n===== Loading {judge_col}: {judge_model_name} =====")

    judge_tokenizer = AutoTokenizer.from_pretrained(judge_model_name)

    if judge_tokenizer.pad_token is None:
        judge_tokenizer.pad_token = judge_tokenizer.eos_token

    judge_model = AutoModelForCausalLM.from_pretrained(
        judge_model_name,
        device_map="auto",
        torch_dtype=torch.float16,
    ).eval()

    scores = []

    for i, row in df.iterrows():
        text = row["text"]

        if not isinstance(text, str) or len(text.strip()) == 0:
            scores.append(None)
            continue

        score = score_with_judge(text, judge_model, judge_tokenizer)
        scores.append(score)

        if (i + 1) % 20 == 0:
            print(f"{judge_col}: {i + 1}/{len(df)}")

    df[judge_col] = scores

    del judge_model
    del judge_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

# Create tables
judge_cols = list(JUDGE_MODELS.keys())
df["judge_avg"] = df[judge_cols].mean(axis=1).round(2)

# p-value/top_p
p_table = (
    df.groupby("top_p")[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

# directory-level comparison
dir_table = (
    df.groupby("source_dir")[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

# model + p-value comparison
model_p_table = (
    df.groupby(["model", "top_p"])[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

# Save results
df.to_csv("p_value_judge_sample_200.csv", index=False)
p_table.to_csv("p_value_judge_summary_by_top_p.csv")
dir_table.to_csv("p_value_judge_summary_by_directory.csv")
model_p_table.to_csv("p_value_judge_summary_by_model_top_p.csv")

print("\nBy top_p:")
display(p_table)

print("\nBy directory:")
display(dir_table)

print("\nBy model and top_p:")
display(model_p_table)

Loaded rows by directory:
source_dir
/content/content/LLM-passphrase/output_dir_exp2    600
output_dir_exp3                                    600
/content/content/LLM-passphrase/output_dir_exp4    600
Name: count, dtype: int64

Detected top_p values:
source_dir
/content/content/LLM-passphrase/output_dir_exp2    [0.95]
/content/content/LLM-passphrase/output_dir_exp4     [1.0]
output_dir_exp3                                    [0.99]
Name: top_p, dtype: object
/content/content/LLM-passphrase/output_dir_exp2: sampled 50
output_dir_exp3: sampled 50
/content/content/LLM-passphrase/output_dir_exp4: sampled 50

Final sample distribution:
source_dir
/content/content/LLM-passphrase/output_dir_exp2    50
output_dir_exp3                                    50
/content/content/LLM-passphrase/output_dir_exp4    50
Name: count, dtype: int64

Top-p per directory:
source_dir
/content/content/LLM-passphrase/output_dir_exp2    [0.95]
/content/content/LLM-passphrase/output_dir_exp4     [1.0]
output_dir_e

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

judge_mistral: 20/150
judge_mistral: 40/150
judge_mistral: 60/150
judge_mistral: 80/150
judge_mistral: 100/150
judge_mistral: 120/150
judge_mistral: 140/150

===== Loading judge_llama3: meta-llama/Meta-Llama-3-8B-Instruct =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

judge_llama3: 20/150
judge_llama3: 40/150
judge_llama3: 60/150
judge_llama3: 80/150
judge_llama3: 100/150
judge_llama3: 120/150
judge_llama3: 140/150

===== Loading judge_gemma: google/gemma-7b-it =====


...

,judge_mistral,judge_llama3,judge_gemma,judge_avg
top_p,,,,
0.95,3.00,3.26,3.06,3.11
0.99,2.88,3.32,3.04,3.08
1.00,2.96,3.24,3.16,3.12



By directory:


,judge_mistral,judge_llama3,judge_gemma,judge_avg
source_dir,,,,
/content/content/LLM-passphrase/output_dir_exp2,3.00,3.26,3.06,3.11
/content/content/LLM-passphrase/output_dir_exp4,2.96,3.24,3.16,3.12
output_dir_exp3,2.88,3.32,3.04,3.08



By model and top_p:


judge_mistral  judge_llama3  judge_gemma  judge_avg
model       top_p                                                     
Llama-2-13b 0.95            3.29          3.14         3.29       3.24
            0.99            3.00          3.29         3.00       3.10
            1.00            3.14          3.00         3.29       3.14
Llama-2-7b  0.95            2.80          3.40         3.00       3.06
            0.99            3.00          3.00         3.20       3.07
            1.00            3.00          3.20         3.40       3.20
Llama-3-8B  0.95            3.00          3.09         3.00       3.03
            0.99            2.64          3.36         3.00       3.00
            1.00            3.00          3.18         3.27       3.15
Mistral-7B  0.95            3.18          3.36         3.18       3.24
            0.99            3.27          3.36         3.09       3.24
            1.00            3.36          3.55         3.18       3.36
Qwen2.5-14B 0.95            2.67          3.25         2.83       2.92
            0.99            2.58          3.17         2.92       2.89
            1.00            2.50          3.00         2.92       2.81
gemma-7b    0.95            3.25          3.50         3.25       3.33
            0.99            3.00          4.00         3.25       3.42
            1.00            2.75          3.75         3.00       3.16

### Extension 3: Test changes in temp but keep p value constant

Using experiments 1.2 1.5 and 1.8 for consisent temp value of 1.2

In [121]:
# Module load again if running separately
import os, json, re, gc
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Directories and parameters
OUTPUT_DIRS = [
    "output_dir_exp3",
    "output_dir_exp6",
    "output_dir_exp9.1",
]

SAMPLE_PER_EXP = 50

JUDGE_MODELS = {
    "judge_mistral": "mistralai/Mistral-7B-Instruct-v0.2",
    "judge_llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
    "judge_gemma": "google/gemma-7b-it",
}

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

#Filtering
END_TOKENS = ["</s>", "<|im_end|>", "<|endoftext|>", "<|eot_id|>", "<eos>"]

def clean(text):
    text = "" if text is None else str(text)
    for tok in END_TOKENS:
        text = text.replace(tok, "")
    text = re.sub(r"^(Sure|Certainly|Here'?s|Okay|Ok)[:,]?\s*", "", text, flags=re.I)
    text = re.sub(r"Give the summary of a story.*", "", text, flags=re.I)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

rows = []

# Data load
for output_dir in OUTPUT_DIRS:
    if not os.path.exists(output_dir):
        print(f"Skipping missing folder: {output_dir}")
        continue

    for root, _, files in os.walk(output_dir):
        for file in files:
            if not file.endswith(".json"):
                continue

            with open(os.path.join(root, file), "r") as f:
                data = json.load(f)

            for k, item in data.items():
                if k == "num_entries":
                    continue

                raw_model = item.get("model")
                model = MODEL_NAME_MAP.get(raw_model, raw_model)

                rows.append({
                    "source_dir": output_dir,
                    "folder": os.path.basename(root),
                    "model": model,
                    "text": clean(item.get("out_sent", "")),
                    "entropy": item.get("sent_ent"),
                    "temp": float(item.get("temp")),
                    "top_p": float(item.get("top_p")),
                    "bot_q": item.get("bot_q"),
                })

df_all = pd.DataFrame(rows)

if df_all.empty:
    raise ValueError("No JSON outputs found. Check OUTPUT_DIRS.")

print("Loaded rows by directory:")
print(df_all["source_dir"].value_counts())

print("\nDetected temperatures:")
print(df_all.groupby("source_dir")["temp"].unique())

print("\nDetected top_p values:")
print(df_all.groupby("source_dir")["top_p"].unique())

# sample 50 per directory
samples = []

for exp_dir in OUTPUT_DIRS:
    df_subset = df_all[df_all["source_dir"] == exp_dir]

    if df_subset.empty:
        print(f"Skipping empty: {exp_dir}")
        continue

    sample_n = min(SAMPLE_PER_EXP, len(df_subset))
    sampled = df_subset.sample(sample_n, random_state=42)
    samples.append(sampled)

    print(f"{exp_dir}: sampled {sample_n}")

df = pd.concat(samples, ignore_index=True)

print("\nFinal sample distribution:")
print(df["source_dir"].value_counts())

print("\nTemperature per directory:")
print(df.groupby("source_dir")["temp"].unique())

print("\nTop-p per directory:")
print(df.groupby("source_dir")["top_p"].unique())

# same prompt for judge
JUDGE_PROMPT = """<s>[INST]
You are a hacker trying to guess the passphrase to log in.
Rate how difficult this passphrase is to guess.

Return ONLY:
Score: X

Where X is 1 to 5.

Passphrase: {text}
[/INST]"""

def score_with_judge(text, judge_model, judge_tokenizer):
    prompt = JUDGE_PROMPT.format(text=text)
    inputs = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)

    outputs = judge_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=judge_tokenizer.eos_token_id,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = judge_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    match = re.search(r"Score\s*:\s*([1-5])", decoded, flags=re.I)
    if not match:
        match = re.search(r"\b([1-5])\b", decoded)

    return int(match.group(1)) if match else None

# run the judges
for judge_col, judge_model_name in JUDGE_MODELS.items():
    print(f"\n===== Loading {judge_col}: {judge_model_name} =====")

    judge_tokenizer = AutoTokenizer.from_pretrained(judge_model_name)

    if judge_tokenizer.pad_token is None:
        judge_tokenizer.pad_token = judge_tokenizer.eos_token

    judge_model = AutoModelForCausalLM.from_pretrained(
        judge_model_name,
        device_map="auto",
        torch_dtype=torch.float16,
    ).eval()

    scores = []

    for i, row in df.iterrows():
        text = row["text"]

        if not isinstance(text, str) or len(text.strip()) == 0:
            scores.append(None)
            continue

        score = score_with_judge(text, judge_model, judge_tokenizer)
        scores.append(score)

        if (i + 1) % 20 == 0:
            print(f"{judge_col}: {i + 1}/{len(df)}")

    df[judge_col] = scores

    del judge_model
    del judge_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

# final results
judge_cols = list(JUDGE_MODELS.keys())
df["judge_avg"] = df[judge_cols].mean(axis=1).round(2)

temp_table = (
    df.groupby("temp")[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

dir_table = (
    df.groupby("source_dir")[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

model_temp_table = (
    df.groupby(["model", "temp"])[judge_cols + ["judge_avg"]]
    .mean()
    .round(2)
)

# save results
df.to_csv("temperature_judge_sample_150.csv", index=False)
temp_table.to_csv("temperature_judge_summary_by_temp.csv")
dir_table.to_csv("temperature_judge_summary_by_directory.csv")
model_temp_table.to_csv("temperature_judge_summary_by_model_temp.csv")

print("\nBy temperature:")
display(temp_table)

print("\nBy directory:")
display(dir_table)

print("\nBy model and temperature:")
display(model_temp_table)

Loaded rows by directory:
source_dir
output_dir_exp3      600
output_dir_exp6      600
output_dir_exp9.1    600
Name: count, dtype: int64

Detected temperatures:
source_dir
output_dir_exp3      [1.0]
output_dir_exp6      [1.2]
output_dir_exp9.1    [1.4]
Name: temp, dtype: object

Detected top_p values:
source_dir
output_dir_exp3      [0.99]
output_dir_exp6      [0.99]
output_dir_exp9.1    [0.99]
Name: top_p, dtype: object
output_dir_exp3: sampled 50
output_dir_exp6: sampled 50
output_dir_exp9.1: sampled 50

Final sample distribution:
source_dir
output_dir_exp3      50
output_dir_exp6      50
output_dir_exp9.1    50
Name: count, dtype: int64

Temperature per directory:
source_dir
output_dir_exp3      [1.0]
output_dir_exp6      [1.2]
output_dir_exp9.1    [1.4]
Name: temp, dtype: object

Top-p per directory:
source_dir
output_dir_exp3      [0.99]
output_dir_exp6      [0.99]
output_dir_exp9.1    [0.99]
Name: top_p, dtype: object

===== Loading judge_mistral: mistralai/Mistral-7B-Instruct-v

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

judge_mistral: 20/150
judge_mistral: 40/150
judge_mistral: 60/150
judge_mistral: 80/150
judge_mistral: 100/150
judge_mistral: 120/150
judge_mistral: 140/150

===== Loading judge_llama3: meta-llama/Meta-Llama-3-8B-Instruct =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

judge_llama3: 20/150
judge_llama3: 40/150
judge_llama3: 60/150
judge_llama3: 80/150
judge_llama3: 100/150
judge_llama3: 120/150
judge_llama3: 140/150

===== Loading judge_gemma: google/gemma-7b-it =====


...

,judge_mistral,judge_llama3,judge_gemma,judge_avg
temp,,,,
1.0,2.90,3.07,3.02,2.99
1.2,3.06,3.20,3.04,3.09
1.4,3.26,3.09,3.22,3.22



By directory:


,judge_mistral,judge_llama3,judge_gemma,judge_avg
source_dir,,,,
output_dir_exp3,2.90,3.07,3.02,2.99
output_dir_exp6,3.06,3.20,3.04,3.09
output_dir_exp9.1,3.26,3.09,3.22,3.22



By model and temperature:


judge_mistral  judge_llama3  judge_gemma  judge_avg
model       temp                                                     
Llama-2-13b 1.0            3.00          3.00         3.00       2.98
            1.2            3.14          3.43         3.14       3.24
            1.4            3.71          3.00         3.29       3.48
Llama-2-7b  1.0            3.00          3.00         3.00       2.97
            1.2            3.00          3.00         2.80       2.90
            1.4            3.00          3.00         3.00       3.00
Llama-3-8B  1.0            2.73          3.09         3.00       2.94
            1.2            3.00          3.10         3.00       3.03
            1.4            3.36          3.18         3.45       3.33
Mistral-7B  1.0            3.27          3.10         3.09       3.15
            1.2            3.36          3.33         3.18       3.29
            1.4            3.73          3.38         3.27       3.47
Qwen2.5-14B 1.0            2.58          3.00         2.92       2.83
            1.2            2.67          3.00         2.92       2.86
            1.4            2.50          2.91         2.82       2.71
gemma-7b    1.0            3.00          3.25         3.25       3.16
            1.2            3.50          3.50         3.25       3.42
            1.4            3.50          3.00         3.75       3.58

## More experiments added based on time

## Experiment 1 extended: Entropy through prrompting

Average entropy of output using the base
prompt, prompt containing low entropy examples and
prompt containing high entropy examples, with temperature of 1.0 and top-p of 1.0.

#### Load modules again if running separately and parameters are here

In [ ]:
import pw_utils
from model_manager import load_model_and_tokenizer
from prompt_manager import create_convs
from main_sample import generate_toks, process_out

import torch
import gc

models = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

prompts = {
    "base": "6",
    "low_ent": "206",
    "high_ent": "306",
}

N = 100
bsz = 1
temperature = 1.0
top_p = 1.0
bot_q = 0.0
sd = 100

In [10]:


for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    for prompt_name, prompt_id in prompts.items():
        convs, quote_label = create_convs(prompt_id)
        out_dir = f"./output_dir/{model_key}_table9_{prompt_name}"

        for conv in convs:
            input_toks = tokenizer.apply_chat_template(
                conv,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )

            if isinstance(input_toks, dict):
                input_toks = input_toks["input_ids"]
            elif hasattr(input_toks, "input_ids"):
                input_toks = input_toks.input_ids

            input_toks = input_toks.to(model.device).repeat(bsz, 1)

            print(f"{model_key}: {prompt_name}, generating {N} samples")

            for i in range(N):
                generated_toks, out_ents = generate_toks(
                    input_toks,
                    model,
                    tokenizer,
                    red_list=None,
                    end_token_ids=None,
                    temperature=temperature,
                    bot_q=bot_q,
                    top_p=top_p,
                    seed=sd + i,
                )

                out = process_out(input_toks, tokenizer, generated_toks, out_ents)

                pw_utils.logger(
                    tokenizer,
                    out_dir,
                    model_name,
                    in_str=quote_label,
                    toks_ents_str=out,
                    temperature=temperature,
                    top_p=top_p,
                    bot_q=bot_q,
                )

                if (i + 1) % 25 == 0:
                    print(f"{model_key}, {prompt_name}: {i + 1}/{N}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Qwen2.5-14B: base, generating 100 samples
Qwen2.5-14B, base: 25/100
Qwen2.5-14B, base: 50/100
Qwen2.5-14B, base: 75/100
Qwen2.5-14B, base: 100/100
Qwen2.5-14B: low_ent, generating 100 samples
Qwen2.5-14B, low_ent: 25/100
Qwen2.5-14B, low_ent: 50/100
Qwen2.5-14B, low_ent: 75/100
Qwen2.5-14B, low_ent: 100/100
Qwen2.5-14B: high_ent, generating 100 samples
Qwen2.5-14B, high_ent: 25/100
Qwen2.5-14B, high_ent: 50/100
Qwen2.5-14B, high_ent: 75/100
Qwen2.5-14B, high_ent: 100/100

===== Loading Mistral-7B =====


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Loaded: Llama-2-13b-chat-hf
Llama-2-13b: base, generating 100 samples
Llama-2-13b, base: 25/100
Llama-2-13b, base: 50/100
Llama-2-13b, base: 75/100
Llama-2-13b, base: 100/100
Llama-2-13b: low_ent, generating 100 samples
Llama-2-13b, low_ent: 25/100
Llama-2-13b, low_ent: 50/100
Llama-2-13b, low_ent: 75/100
Llama-2-13b, low_ent: 100/100
Llama-2-13b: high_ent, generating 100 samples
Llama-2-13b, high_ent: 25/100
Llama-2-13b, high_ent: 50/100
Llama-2-13b, high_ent: 75/100
Llama-2-13b, high_ent: 100/100


In [15]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

PROMPT_ORDER = ["base", "low_ent", "high_ent"]

rows = []

for root, dirs, files in os.walk("output_dir"):
    if "table9_" not in root.lower():
        continue

    prompt_name = root.split("table9_")[-1]

    for file in files:
        if not file.endswith(".json"):
            continue

        with open(os.path.join(root, file), "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            raw_model = item.get("model")
            model = MODEL_NAME_MAP.get(raw_model, raw_model)

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            rows.append({
                "model": model,
                "prompt": prompt_name,
                "entropy": float(entropy),
            })

df9 = pd.DataFrame(rows)

table9 = df9.pivot_table(
    index="model",
    columns="prompt",
    values="entropy",
    aggfunc="mean"
).round(1)

table9 = table9.reindex([m for m in MODEL_ORDER if m in table9.index])
table9 = table9[[c for c in PROMPT_ORDER if c in table9.columns]]

table9.to_csv("table9_entropy_prompt_comparison.csv")

table9

prompt,base,low_ent,high_ent
model,,,
Qwen2.5-14B,13.1,13.1,18.0
Mistral-7B,52.7,30.2,59.7
gemma-7b,47.5,30.1,35.0
Llama-2-7b,34.9,21.9,34.2
Llama-3-8B,39.1,25.5,37.6
Llama-2-13b,43.0,19.8,39.7


### Experiment Top-q truncation

Average entropy of output at various q values. Sample size of 100 using base prompt with temperature of 1.0 and top-p of 0.95

#### Backup the utils file

We changed the utils file for added safety as there was a cuda error thrown initially due to errors with faulty sampling logic

In [122]:
# backup first
!cp pw_utils.py pw_utils_backup.py

In [123]:
%%writefile pw_utils_patch.py
import torch

def pw_sample_top_p_safe(probs, q, p, eos_tok_id, end_token_ids):
    probs_sort, probs_idx = torch.sort(probs, dim=-1, descending=True)

    # cumulative probabilities
    probs_sum = torch.cumsum(probs_sort, dim=-1)

    # bottom-q truncation: remove lowest cumulative mass from the head/tail logic safely
    if q > 0.0:
        l_mask = probs_sum < q
        probs_sort = probs_sort.masked_fill(l_mask, 0.0)

    # top-p truncation
    probs_sum = torch.cumsum(probs_sort, dim=-1)
    p_mask = probs_sum - probs_sort > p
    probs_sort = probs_sort.masked_fill(p_mask, 0.0)

    # safety repair
    row_sums = probs_sort.sum(dim=-1, keepdim=True)

    bad_rows = (
        torch.isnan(row_sums).squeeze(-1)
        | torch.isinf(row_sums).squeeze(-1)
        | (row_sums.squeeze(-1) <= 0)
    )

    if bad_rows.any():
        probs_sort[bad_rows] = 0.0
        probs_sort[bad_rows, 0] = 1.0
        row_sums = probs_sort.sum(dim=-1, keepdim=True)

    probs_sort = probs_sort / row_sums

    next_token_pos = torch.multinomial(probs_sort, num_samples=1)
    sampled_prob = torch.gather(probs_sort, -1, next_token_pos).clamp_min(1e-12)
    token_ent = -torch.log2(sampled_prob)

    next_token = torch.gather(probs_idx, -1, next_token_pos)

    if end_token_ids:
        end_token_tensor = torch.tensor(end_token_ids, device=next_token.device)
        mask = torch.isin(next_token, end_token_tensor)
        next_token[mask] = eos_tok_id

    return next_token, token_ent

Writing pw_utils_patch.py


#### Load modules from updated utils file

In [124]:
import pw_utils
from pw_utils_patch import pw_sample_top_p_safe

pw_utils.pw_sample_top_p = pw_sample_top_p_safe

import os, gc, torch
import pw_utils

from model_manager import load_model_and_tokenizer
from prompt_manager import create_convs
from main_sample import generate_toks, process_out


#### Set parametes and models

In [125]:
in_prompt = "6"
temperature = 1.0
top_p = 0.95
bot_q_values = [0.0, 0.05, 0.20, 0.35]

models = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

bsz = 1
N = 100
sd = 100

In [126]:

for model_key in models:
    print(f"\n===== Loading {model_key} =====")

    model_name, model, tokenizer = load_model_and_tokenizer(model_key)
    print("Loaded:", model_name)

    convs, quote_label = create_convs(in_prompt)

    for bot_q in bot_q_values:
        out_dir = f"./output_dir_table4/{model_key}_q{bot_q}"

        for conv in convs:
            input_toks = tokenizer.apply_chat_template(
                conv,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )

            if isinstance(input_toks, dict):
                input_toks = input_toks["input_ids"]
            elif hasattr(input_toks, "input_ids"):
                input_toks = input_toks.input_ids

            input_toks = input_toks.to(model.device).repeat(bsz, 1)

            print(f"{model_key}: q={bot_q}, generating {N}")

            for i in range(N):
                generated_toks, out_ents = generate_toks(
                    input_toks,
                    model,
                    tokenizer,
                    red_list=None,
                    end_token_ids=None,
                    temperature=temperature,
                    bot_q=bot_q,
                    top_p=top_p,
                    seed=sd + i,
                )

                out = process_out(input_toks, tokenizer, generated_toks, out_ents)

                pw_utils.logger(
                    tokenizer,
                    out_dir,
                    model_name,
                    in_str=quote_label,
                    toks_ents_str=out,
                    temperature=temperature,
                    top_p=top_p,
                    bot_q=bot_q,
                )

                if (i + 1) % 20 == 0:
                    print(f"{model_key}, q={bot_q}: {i + 1}/{N}")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    print(f"Finished {model_key}")


===== Loading Qwen2.5-14B =====


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loaded: Qwen2.5-14B-Instruct
Qwen2.5-14B: q=0.0, generating 100
Qwen2.5-14B, q=0.0: 20/100
Qwen2.5-14B, q=0.0: 40/100
Qwen2.5-14B, q=0.0: 60/100
Qwen2.5-14B, q=0.0: 80/100
Qwen2.5-14B, q=0.0: 100/100
Qwen2.5-14B: q=0.05, generating 100
Qwen2.5-14B, q=0.05: 20/100
Qwen2.5-14B, q=0.05: 40/100
Qwen2.5-14B, q=0.05: 60/100
Qwen2.5-14B, q=0.05: 80/100
Qwen2.5-14B, q=0.05: 100/100
Qwen2.5-14B: q=0.2, generating 100
Qwen2.5-14B, q=0.2: 20/100
Qwen2.5-14B, q=0.2: 40/100
Qwen2.5-14B, q=0.2: 60/100
Qwen2.5-14B, q=0.2: 80/100
Qwen2.5-14B, q=0.2: 100/100
Qwen2.5-14B: q=0.35, generating 100
Qwen2.5-14B, q=0.35: 20/100
Qwen2.5-14B, q=0.35: 40/100
Qwen2.5-14B, q=0.35: 60/100
Qwen2.5-14B, q=0.35: 80/100
Qwen2.5-14B, q=0.35: 100/100
Finished Qwen2.5-14B

===== Loading Mistral-7B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Mistral-7B-Instruct-v0.2
Mistral-7B: q=0.0, generating 100
Mistral-7B, q=0.0: 20/100
Mistral-7B, q=0.0: 40/100
Mistral-7B, q=0.0: 60/100
Mistral-7B, q=0.0: 80/100
Mistral-7B, q=0.0: 100/100
Mistral-7B: q=0.05, generating 100
Mistral-7B, q=0.05: 20/100
Mistral-7B, q=0.05: 40/100
Mistral-7B, q=0.05: 60/100
Mistral-7B, q=0.05: 80/100
Mistral-7B, q=0.05: 100/100
Mistral-7B: q=0.2, generating 100
Mistral-7B, q=0.2: 20/100
Mistral-7B, q=0.2: 40/100
Mistral-7B, q=0.2: 60/100
Mistral-7B, q=0.2: 80/100
Mistral-7B, q=0.2: 100/100
Mistral-7B: q=0.35, generating 100
Mistral-7B, q=0.35: 20/100
Mistral-7B, q=0.35: 40/100
Mistral-7B, q=0.35: 60/100
Mistral-7B, q=0.35: 80/100
Mistral-7B, q=0.35: 100/100
Finished Mistral-7B

===== Loading gemma-7b =====


...

Loaded: Llama-2-7b-chat-hf
Llama-2-7b: q=0.0, generating 100
Llama-2-7b, q=0.0: 20/100
Llama-2-7b, q=0.0: 40/100
Llama-2-7b, q=0.0: 60/100
Llama-2-7b, q=0.0: 80/100
Llama-2-7b, q=0.0: 100/100
Llama-2-7b: q=0.05, generating 100
Llama-2-7b, q=0.05: 20/100
Llama-2-7b, q=0.05: 40/100
Llama-2-7b, q=0.05: 60/100
Llama-2-7b, q=0.05: 80/100
Llama-2-7b, q=0.05: 100/100
Llama-2-7b: q=0.2, generating 100
Llama-2-7b, q=0.2: 20/100
Llama-2-7b, q=0.2: 40/100
Llama-2-7b, q=0.2: 60/100
Llama-2-7b, q=0.2: 80/100
Llama-2-7b, q=0.2: 100/100
Llama-2-7b: q=0.35, generating 100
Llama-2-7b, q=0.35: 20/100
Llama-2-7b, q=0.35: 40/100
Llama-2-7b, q=0.35: 60/100
Llama-2-7b, q=0.35: 80/100
Llama-2-7b, q=0.35: 100/100
Finished Llama-2-7b

===== Loading Llama-3-8B =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: Meta-Llama-3-8B-Instruct
Llama-3-8B: q=0.0, generating 100
Llama-3-8B, q=0.0: 20/100
Llama-3-8B, q=0.0: 40/100
Llama-3-8B, q=0.0: 60/100
Llama-3-8B, q=0.0: 80/100
Llama-3-8B, q=0.0: 100/100
Llama-3-8B: q=0.05, generating 100
Llama-3-8B, q=0.05: 20/100
Llama-3-8B, q=0.05: 40/100
Llama-3-8B, q=0.05: 60/100
Llama-3-8B, q=0.05: 80/100
Llama-3-8B, q=0.05: 100/100
Llama-3-8B: q=0.2, generating 100
Llama-3-8B, q=0.2: 20/100
Llama-3-8B, q=0.2: 40/100
Llama-3-8B, q=0.2: 60/100
Llama-3-8B, q=0.2: 80/100
Llama-3-8B, q=0.2: 100/100
Llama-3-8B: q=0.35, generating 100
Llama-3-8B, q=0.35: 20/100
Llama-3-8B, q=0.35: 40/100
Llama-3-8B, q=0.35: 60/100
Llama-3-8B, q=0.35: 80/100
Llama-3-8B, q=0.35: 100/100
Finished Llama-3-8B

===== Loading Llama-2-13b =====


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loaded: Llama-2-13b-chat-hf
Llama-2-13b: q=0.0, generating 100
Llama-2-13b, q=0.0: 20/100
Llama-2-13b, q=0.0: 40/100
Llama-2-13b, q=0.0: 60/100
Llama-2-13b, q=0.0: 80/100
Llama-2-13b, q=0.0: 100/100
Llama-2-13b: q=0.05, generating 100
Llama-2-13b, q=0.05: 20/100
Llama-2-13b, q=0.05: 40/100
Llama-2-13b, q=0.05: 60/100
Llama-2-13b, q=0.05: 80/100
Llama-2-13b, q=0.05: 100/100
Llama-2-13b: q=0.2, generating 100
Llama-2-13b, q=0.2: 20/100
Llama-2-13b, q=0.2: 40/100
Llama-2-13b, q=0.2: 60/100
Llama-2-13b, q=0.2: 80/100
Llama-2-13b, q=0.2: 100/100
Llama-2-13b: q=0.35, generating 100
Llama-2-13b, q=0.35: 20/100
Llama-2-13b, q=0.35: 40/100
Llama-2-13b, q=0.35: 60/100
Llama-2-13b, q=0.35: 80/100
Llama-2-13b, q=0.35: 100/100
Finished Llama-2-13b


In [127]:
import os, json, re
import pandas as pd

MODEL_NAME_MAP = {
    "Qwen2.5-14B-Instruct": "Qwen2.5-14B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "gemma-7b-it": "gemma-7b",
    "Llama-2-7b-chat-hf": "Llama-2-7b",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "Llama-2-13b-chat-hf": "Llama-2-13b",
}

MODEL_ORDER = [
    "Qwen2.5-14B",
    "Mistral-7B",
    "gemma-7b",
    "Llama-2-7b",
    "Llama-3-8B",
    "Llama-2-13b",
]

rows = []

for root, _, files in os.walk("output_dir_table4"):
    if "_q" not in root:
        continue

    q_match = re.search(r"_q([0-9.]+)", root)
    if not q_match:
        continue

    q_val = float(q_match.group(1))

    for file in files:
        if not file.endswith(".json"):
            continue

        with open(os.path.join(root, file), "r") as f:
            data = json.load(f)

        for k, item in data.items():
            if k == "num_entries":
                continue

            entropy = item.get("sent_ent")
            if entropy is None:
                continue

            model = MODEL_NAME_MAP.get(item.get("model"), item.get("model"))

            rows.append({
                "model": model,
                "q": q_val,
                "entropy": float(entropy),
            })

df = pd.DataFrame(rows)

table4 = df.pivot_table(
    index="model",
    columns="q",
    values="entropy",
    aggfunc="mean"
).round(1)

table4 = table4.reindex([m for m in MODEL_ORDER if m in table4.index])

table4.to_csv("table4_results.csv")

table4

q,0.00,0.05,0.20,0.35
model,,,,
Qwen2.5-14B,10.8,10.8,10.8,13.8
Mistral-7B,46.0,47.1,53.5,54.7
gemma-7b,39.3,39.3,41.2,46.8
Llama-2-7b,31.1,31.1,34.1,38.0
Llama-3-8B,34.9,35.4,38.3,42.7
Llama-2-13b,37.3,38.0,42.3,46.7
